In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv(
    "../data/equlens_engineered_features.csv",
    index_col=0,
    parse_dates=True
)

data.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Daily_Return,Price_Change,Price_Change_Pct,...,High_Volume_Flag,RSI_14,MACD,MACD_Signal,MACD_Histogram,Price_Range,Open_Close_Return,High_Low_Pct,Target_Return_1D,Target_Up
Date,,,,,,,,,,,,,,,,,,,,,
2024-11-25 00:00:00-05:00,229.792416,231.569513,228.084806,231.192245,90152800,0.0,0.0,0.013051,1.399830,0.006092,...,1,69.706490,0.658897,0.033839,0.625058,0.015073,0.006092,0.015278,0.009404,1
2024-11-26 00:00:00-05:00,231.648908,233.872775,231.648908,233.366440,45986200,0.0,0.0,0.009404,1.717531,0.007414,...,0,74.215693,1.132539,0.253579,0.878960,0.009530,0.007414,0.009600,-0.000553,0
2024-11-27 00:00:00-05:00,232.780704,233.991915,232.125455,233.237381,33498400,0.0,0.0,-0.000553,0.456677,0.001962,...,0,68.024384,1.480424,0.498948,0.981476,0.008002,0.001962,0.008041,0.010216,1
2024-11-29 00:00:00-05:00,233.118254,236.096639,232.284309,235.620102,28481400,0.0,0.0,0.010216,2.501848,0.010732,...,0,72.073274,1.926187,0.784396,1.141791,0.016180,0.010732,0.016412,0.009523,1
2024-12-02 00:00:00-05:00,235.560538,239.055167,235.451330,237.863815,48137100,0.0,0.0,0.009523,2.303277,0.009778,...,0,83.362354,2.432467,1.114010,1.318457,0.015151,0.009778,0.015306,0.012772,1


In [3]:
print("Dataset shape:", data.shape)
print("Columns:")
print(data.columns.tolist())

Dataset shape: (452, 33)
Columns:
['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Daily_Return', 'Price_Change', 'Price_Change_Pct', 'High_Low_Range', 'MA_5', 'MA_20', 'MA_50', 'Price_to_MA20', 'Price_to_MA50', 'Return_5D', 'Return_20D', 'Momentum_10D', 'Volatility_5D', 'Volatility_20D', 'Volume_MA_20', 'Volume_Ratio', 'High_Volume_Flag', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Price_Range', 'Open_Close_Return', 'High_Low_Pct', 'Target_Return_1D', 'Target_Up']


In [4]:
data.isnull().sum()

Open                 0
High                 0
Low                  0
Close                0
Volume               0
Dividends            0
Stock Splits         0
Daily_Return         0
Price_Change         0
Price_Change_Pct     0
High_Low_Range       0
MA_5                 0
MA_20                0
MA_50                0
Price_to_MA20        0
Price_to_MA50        0
Return_5D            0
Return_20D           0
Momentum_10D         0
Volatility_5D        0
Volatility_20D       0
Volume_MA_20         0
Volume_Ratio         0
High_Volume_Flag     0
RSI_14               0
MACD                 0
MACD_Signal          0
MACD_Histogram       0
Price_Range          0
Open_Close_Return    0
High_Low_Pct         0
Target_Return_1D     0
Target_Up            0
dtype: int64

In [5]:
data = data.dropna()

print("Shape after removing missing values:", data.shape)
print("Remaining missing values:", data.isnull().sum().sum())

Shape after removing missing values: (452, 33)
Remaining missing values: 0


In [6]:
print(data["Target_Up"].value_counts())
print(data["Target_Up"].value_counts(normalize=True))

Target_Up
1    243
0    209
Name: count, dtype: int64
Target_Up
1    0.537611
0    0.462389
Name: proportion, dtype: float64


In [7]:
X = data.drop(columns=["Target_Up"])
y = data["Target_Up"]

In [8]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (452, 32)
Target shape: (452,)


In [9]:
print(X.columns.tolist())

['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Daily_Return', 'Price_Change', 'Price_Change_Pct', 'High_Low_Range', 'MA_5', 'MA_20', 'MA_50', 'Price_to_MA20', 'Price_to_MA50', 'Return_5D', 'Return_20D', 'Momentum_10D', 'Volatility_5D', 'Volatility_20D', 'Volume_MA_20', 'Volume_Ratio', 'High_Volume_Flag', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Price_Range', 'Open_Close_Return', 'High_Low_Pct', 'Target_Return_1D']


In [10]:
columns_to_drop = ["Dividends", "Stock Splits"]

X = X.drop(
    columns=[col for col in columns_to_drop if col in X.columns],
    errors="ignore"
)

print("Final feature columns:")
print(X.columns.tolist())

Final feature columns:
['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'Price_Change', 'Price_Change_Pct', 'High_Low_Range', 'MA_5', 'MA_20', 'MA_50', 'Price_to_MA20', 'Price_to_MA50', 'Return_5D', 'Return_20D', 'Momentum_10D', 'Volatility_5D', 'Volatility_20D', 'Volume_MA_20', 'Volume_Ratio', 'High_Volume_Flag', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Price_Range', 'Open_Close_Return', 'High_Low_Pct', 'Target_Return_1D']


In [11]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [12]:
print("Training period:", X_train.index.min(), "to", X_train.index.max())
print("Testing period:", X_test.index.min(), "to", X_test.index.max())

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Training period: 2024-11-25 00:00:00-05:00 to 2026-05-06 00:00:00-04:00
Testing period: 2026-05-07 00:00:00-04:00 to 2026-09-16 00:00:00-04:00
X_train shape: (361, 30)
X_test shape: (91, 30)
y_train shape: (361,)
y_test shape: (91,)


In [13]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training target distribution:
Target_Up
1    0.531856
0    0.468144
Name: proportion, dtype: float64

Testing target distribution:
Target_Up
1    0.56044
0    0.43956
Name: proportion, dtype: float64


In [14]:
train_data = X_train.copy()
train_data["Target_Up"] = y_train

test_data = X_test.copy()
test_data["Target_Up"] = y_test

In [15]:
train_data.to_csv("../data/train_data.csv")
test_data.to_csv("../data/test_data.csv")

In [16]:
print("Training data saved:", train_data.shape)
print("Testing data saved:", test_data.shape)

Training data saved: (361, 31)
Testing data saved: (91, 31)
